# 05 RAGAS Evaluation

Generation CSV 파일을 입력으로 사용해 RAGAS 평가를 수행합니다. row 단위 RAGAS 결과는 로컬 detail CSV에 저장하고, `use_weave=True`일 때 Weave에는 row 단위 case 로그와 summary 로그를 남깁니다.


## Git Clone

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/songhahyun/finance-terms-rag-chatbot.git'
REPO_BRANCH = 'dev'
REPO_DIR = Path('/content/finance-terms-rag-chatbot')

In [ ]:
!git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
print('cwd =', Path.cwd())

In [ ]:
# Python deps
!pip install -q -U pip
!pip install -q -r requirements.txt

## Fetch API Keys

In [ ]:
from google.colab import userdata

REQUIRED_SECRETS = [
    "OPENAI_API_KEY",
    "NCP_APIGW_API_KEY",
    "CLOVASTUDIO_API_KEY",
    "HF_TOKEN",
    "WANDB_API_KEY",
]

missing = []

for name in REQUIRED_SECRETS:
    value = userdata.get(name)
    if value:
        os.environ[name] = value
    else:
        missing.append(name)

if missing:
    raise RuntimeError(
        "Colab Secrets에 다음 값을 추가하고 Notebook access를 켜세요: "
        + ", ".join(missing)
    )

## Run evaluation

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")

Project root: d:\AI\projects\finance-terms-rag-chatbot


In [ ]:
from src.common.config import get_settings
from src.evaluation.ragas_pipeline import run_ragas_evaluations

settings = get_settings()

GENERATION_OUTPUT_DIR = ROOT / "data" / "eval" / "outputs" / "generation_002_language_validator"
RAGAS_OUTPUT_DIR = ROOT / "data" / "eval" / "outputs" / "ragas_002_language_validator"
RAGAS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAMES = [
    "dense_clova_bge-m3",
    "hybrid_clova_bge-m3",
    "hybrid_openai_text-embedding-3-small",
]
GENERATED_CSV_PATHS = [GENERATION_OUTPUT_DIR / f"{name}.csv" for name in EXPERIMENT_NAMES]

CHUNK_JSON_PATH = settings.default_chunk_json_path
JUDGE_MODEL = "gpt-4o-mini"
JUDGE_EMBEDDING_MODEL = "multilingual-e5-large"
MAX_ROWS = 5  # Set to None for the full evaluation.

USE_WEAVE = True
WEAVE_PROJECT = "finance-terms-rag-evaluation"
WEAVE_EXPERIMENT_GROUP = "generation_002_language_validator"
WEAVE_LOG_CONTEXTS = True
WEAVE_PRINT_CALL_LINK = False
RATE_LIMIT_MAX_RETRIES = 20
RATE_LIMIT_SLEEP_SECONDS = 10.0
RATE_LIMIT_MAX_SLEEP_SECONDS = 120.0

missing_csv_paths = [path for path in GENERATED_CSV_PATHS if not path.exists()]
if missing_csv_paths:
    raise FileNotFoundError("Missing generation CSV files: " + ", ".join(str(path) for path in missing_csv_paths))

print("Generation output dir:", GENERATION_OUTPUT_DIR)
print("RAGAS output dir:", RAGAS_OUTPUT_DIR)
print("Generated CSV paths:")
for path in GENERATED_CSV_PATHS:
    print(f"- {path}")
print("Chunk JSON:", CHUNK_JSON_PATH)
print("Judge model:", JUDGE_MODEL)
print("Judge embedding model:", JUDGE_EMBEDDING_MODEL)
print("Max rows:", MAX_ROWS)
print("Use Weave:", USE_WEAVE)
print("Weave project:", WEAVE_PROJECT)
print("Weave experiment group:", WEAVE_EXPERIMENT_GROUP)
print("Rate limit max retries:", RATE_LIMIT_MAX_RETRIES)


In [ ]:
combined_summary_path = RAGAS_OUTPUT_DIR / "ragas_experiment_summary.csv"

detail_outputs, summary_df = run_ragas_evaluations(
    generated_csv_paths=GENERATED_CSV_PATHS,
    chunk_json_path=CHUNK_JSON_PATH,
    output_dir=RAGAS_OUTPUT_DIR,
    output_summary_path=combined_summary_path,
    judge_model=JUDGE_MODEL,
    judge_embedding_model=JUDGE_EMBEDDING_MODEL,
    max_rows=MAX_ROWS,
    use_weave=USE_WEAVE,
    weave_project=WEAVE_PROJECT,
    weave_experiment_group=WEAVE_EXPERIMENT_GROUP,
    weave_log_contexts=WEAVE_LOG_CONTEXTS,
    weave_print_call_link=WEAVE_PRINT_CALL_LINK,
    rate_limit_max_retries=RATE_LIMIT_MAX_RETRIES,
    rate_limit_sleep_seconds=RATE_LIMIT_SLEEP_SECONDS,
    rate_limit_max_sleep_seconds=RATE_LIMIT_MAX_SLEEP_SECONDS,
)

print("Saved RAGAS output files:")
for name, path in detail_outputs.items():
    print(f"- {name}: {path}")
print(f"- combined summary: {combined_summary_path}")

summary_df
